# Lid-Driven Cavity — Steady Navier-Stokes

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/camlab-ethz/TensorMesh/blob/main/notebooks/cavity.ipynb)

The canonical incompressible-flow benchmark: a unit square whose top wall
slides at constant speed while the other three walls stay put. Discretized
with the **LBB-stable Taylor-Hood P2-P1 pair** and linearized by **Picard
iteration** — no SUPG/PSPG stabilization anywhere, because the element pair
is inf-sup stable by construction.

If you have already seen the [Stokes notebook](https://colab.research.google.com/github/camlab-ethz/TensorMesh/blob/main/notebooks/stokes_taylor_hood.ipynb),
this is the same assembler with a convection term added.

⏱️ *A couple of minutes on Colab's free CPU runtime; lower `N_GRID` to speed it up.*

Docs: [Lid-Driven Cavity](https://docs.tensor-mesh.com/example_gallery/fluid/cavity.html) · Source: [`examples/fluid/cavity/cavity.py`](https://github.com/camlab-ethz/TensorMesh/blob/main/examples/fluid/cavity/cavity.py)

In [ ]:
# Install TensorMesh (skipped automatically if it is already available, e.g. a local dev setup).
# The apt line provides the OpenGL utility library that gmsh -- TensorMesh's mesh generator --
# needs at import time; it is a no-op where the library is already present.
import importlib.util
if importlib.util.find_spec("tensormesh") is None:
    !apt-get -qq install -y libglu1-mesa > /dev/null 2>&1 || true
    %pip install -q tensormesh-fem==0.2.0

import contextlib
import os


@contextlib.contextmanager
def quiet():
    """Hide gmsh's meshing log, which is written below Python's stdout.
    Drop the ``with quiet():`` wrapper anywhere to see what the mesher is doing."""
    with open(os.devnull, "w") as null:
        saved = os.dup(1)
        os.dup2(null.fileno(), 1)
        try:
            yield
        finally:
            os.dup2(saved, 1)
            os.close(saved)

## Weak form

$$\rho\,(w\cdot\nabla)u\cdot v + \mu\,\nabla u : \nabla v - p\,\nabla\cdot v - q\,\nabla\cdot u$$

The nonlinear convection $(u\cdot\nabla)u$ is lagged to $(w\cdot\nabla)u$
with $w$ the previous iterate, which makes each step a linear solve.

In [ ]:
import torch

from tensormesh import Condenser, Field, Mesh, MixedElementAssembler


class NavierStokesAssembler(MixedElementAssembler):
    r"""Picard-linearized steady Navier-Stokes:

    .. math::

        \rho\,(w\cdot\nabla)u\cdot v + \mu\,\nabla u : \nabla v
        - p\,\nabla\cdot v - q\,\nabla\cdot u,

    with ``w`` the previous velocity iterate (passed via ``point_data``).
    ``gradu`` is the velocity Jacobian ``[2, 2]``, so the convection term is
    ``gradu @ w`` and the divergence is its trace.
    """

    fields = [
        Field(trial="u", test="v", order=2, components=2),
        Field(trial="p", test="q", order=1),
    ]

    def __post_init__(self, rho=1.0, mu=0.01):
        self.rho = rho
        self.mu = mu

    def forward(self, gradu, p, v, gradv, q, w):
        convection = self.rho * (gradu @ w).dot(v)
        diffusion = self.mu * (gradu * gradv).sum()
        return convection + diffusion \
            - p * gradv.diagonal().sum() \
            - q * gradu.diagonal().sum()

## Mesh and boundary conditions

In [ ]:
RE = 100       # Reynolds number
N_GRID = 30    # mesh resolution; lower to ~20 for a quicker run

# Order-2 geometry: the mesh points *are* the P2 velocity nodes here.
with quiet():
    mesh = Mesh.gen_rectangle(chara_length=1.0 / N_GRID, order=2).double()
assembler = NavierStokesAssembler.from_mesh(mesh, rho=1.0, mu=1.0 / RE)
layout = assembler.layout
print(f"mesh: {mesh.n_points} P2 velocity nodes, "
      f"{layout.n_nodes('p')} P1 pressure nodes, {layout.n_dofs} DOFs")

# Boundary conditions: no-slip everywhere, lid moving at u_x = 1 along the top,
# plus one pinned pressure DOF to remove the constant null space.
is_top = mesh.points[:, 1] > 1.0 - 1e-6

bc_mask = layout.dof_mask("u", mesh.boundary_mask)
bc_mask[layout.dof_index("p", int(layout.node_ids("p")[0]))] = True

bc_val = torch.zeros(layout.n_dofs, dtype=torch.float64)
bc_val[layout.dof_mask("u", is_top, component=0)] = 1.0

condenser = Condenser(bc_mask, bc_val[bc_mask])

## Picard iteration

Reassemble around the current velocity, solve, repeat until the update stops
moving. The `Condenser` is built once and reused every iteration.

In [ ]:
max_iter, tol = 20, 1e-4

sol = torch.zeros(layout.n_dofs, dtype=torch.float64)
sol[bc_mask] = bc_val[bc_mask]

for i in range(max_iter):
    w = layout.split(sol)["u"]  # previous-iterate velocity, [n_points, 2]
    K = assembler(point_data={"w": w})
    f = torch.zeros(layout.n_dofs, dtype=torch.float64)

    K_, f_ = condenser(K, f)
    sol_new = condenser.recover(K_.solve(f_))

    diff = torch.norm(sol_new - sol) / (torch.norm(sol_new) + 1e-8)
    print(f"Picard {i:2d}: relative diff = {diff:.6e}")
    sol = sol_new
    if diff < tol:
        print("Converged!")
        break

In [ ]:
fields = layout.split(sol)
speed = torch.norm(fields["u"], dim=1)
pressure = layout.prolong("p", fields["p"])  # P1 pressure on all P2 mesh points

mesh.plot(
    {"speed": speed, "pressure": pressure},
    save_path="cavity_results.png", show_mesh=False, cmap="jet",
)
import matplotlib.pyplot as plt
plt.close("all")  # mesh.plot leaves its figure open; avoid a duplicate inline render
from IPython.display import Image
Image("cavity_results.png")

## Sanity check

The centreline profile is the standard way to compare against the Ghia et al.
reference data: a negative $u_x$ through most of the cavity depth, turning
positive only in the thin layer driven by the lid.

In [ ]:
# Classic validation: u_x along the vertical centreline (Ghia et al. 1982).
import matplotlib.pyplot as plt

pts = mesh.points
on_centre = (pts[:, 0] - 0.5).abs() < 1.0 / (2 * N_GRID)
y = pts[on_centre, 1]
ux = fields["u"][on_centre, 0]
order = torch.argsort(y)

fig, ax = plt.subplots(figsize=(4.5, 5))
ax.plot(ux[order], y[order], "o-", ms=3)
ax.axvline(0.0, color="gray", lw=0.8)
ax.set_xlabel(r"$u_x$")
ax.set_ylabel("y")
ax.set_title(f"Centreline velocity, Re={RE}")
ax.grid(alpha=0.3)
plt.show()